In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import json
import plotly.graph_objects as go
import logging

import utils
import z_utils
import network

sequence_length = 200
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [3]:
loader = utils.get_m2s_loader(128)

# Sparate the data into train and test
train_loader = []
test_loader = []

for i, data in enumerate(loader):
    if i % 10 == 0:
        test_loader.append(data)
    else:
        train_loader.append(data)

test_df = pd.read_csv('../../Data_processed/Kmedoids_with_Weather_Espike_test.csv')
print(test_df.shape)
print(test_df.head())


(1682533, 19)
       LCLid  energy(kWh/hh)_smoothed  energy(kWh/hh)   mean  median     std  \
0  MAC000051                   0.3409           0.247  0.323    0.21  0.3232   
1  MAC000051                   0.3760           0.318  0.323    0.21  0.3232   
2  MAC000051                   0.3652           0.284  0.323    0.21  0.3232   
3  MAC000051                   0.4082           0.306  0.323    0.21  0.3232   
4  MAC000051                   0.4033           0.337  0.323    0.21  0.3232   

   min    max  gradient  kmedoid_clusters  spike_type  spike_magnitude  \
0  0.0  3.462    0.1614                17           0              1.0   
1  0.0  3.462    0.1614                17           0              1.0   
2  0.0  3.462    0.1614                17           0              1.0   
3  0.0  3.462    0.1614                17           0              1.0   
4  0.0  3.462    0.1614                17           0              1.0   

   temperature  windSpeed  humidity  date_sin  date_cos  tim

In [4]:
class Generator(nn.Module):
    def __init__(self, s_embed_size, c_embed_size, time_size, time_out, stat_size, stat_out, hidden_size):
        super(Generator, self).__init__()

        self.Embedding_spike = nn.Embedding(5, s_embed_size)
        self.Embedding_cluster = nn.Embedding(20, c_embed_size)

        self.time_net = nn.Sequential(
            nn.Linear(4, time_size),
            nn.ReLU(),
            nn.Linear(time_size, time_out),
            nn.ReLU()
        )

        self.statistical_net = nn.Sequential(
            nn.Linear(6, stat_size),
            nn.ReLU(),
            nn.Linear(stat_size, stat_out),
            nn.ReLU()
        )

        self.fc = nn.Sequential(
            nn.Linear(s_embed_size + c_embed_size + time_out + stat_out + 3 + 6 + 1, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )
    
    def forward(self, weather, cluster, time, statistical, spike_type, spike_magnitude):
        sp = self.Embedding_spike(spike_type).squeeze(2)
        c = self.Embedding_cluster(cluster).squeeze(2)
        t = self.time_net(time)
        s = self.statistical_net(statistical)
        out = torch.cat((weather, c, t, s, statistical, sp, spike_magnitude), dim=-1)

        return self.fc(out)

# We will put all the auxiliary information for the Discriminator too, to make it better.
class Discriminator(nn.Module):
    def __init__(self, s_embed_size, c_embed_size, time_size, time_out, stat_size, stat_out, fc1_hidden_size, fc1_out, fc2_hidden_size):
        super(Discriminator, self).__init__()

        self.Embedding_spike = nn.Embedding(5, s_embed_size)
        self.Embedding_cluster = nn.Embedding(20, c_embed_size)

        self.time_net = nn.Sequential(
            nn.Linear(4, time_size),
            nn.ReLU(),
            nn.Linear(time_size, time_out),
            nn.ReLU()
        )

        self.statistical_net = nn.Sequential(
            nn.Linear(6, stat_size),
            nn.ReLU(),
            nn.Linear(stat_size, stat_out),
            nn.ReLU()
        )
        
        self.fc1 = nn.Sequential(
            nn.Linear(s_embed_size + c_embed_size + time_out + stat_out + 3 + 6 + 1 + 1, fc1_hidden_size),
            nn.ReLU(),
            nn.Linear(fc1_hidden_size, fc1_hidden_size),
            nn.ReLU(),
            nn.Linear(fc1_hidden_size, fc1_out),
            nn.ReLU()
        )

        self.fc2 = nn.Sequential(
            nn.Linear(fc1_out + 1, fc2_hidden_size),
            nn.ReLU(),
            nn.Linear(fc2_hidden_size, fc2_hidden_size),
            nn.ReLU(),
            nn.Linear(fc2_hidden_size, 1)
        )

    def forward(self, weather, cluster, time, statistical, spike_type, spike_magnitude, energy):
        sp = self.Embedding_spike(spike_type).squeeze(2)
        c = self.Embedding_cluster(cluster).squeeze(2)
        t = self.time_net(time)
        s = self.statistical_net(statistical)
        out = torch.cat((weather, c, t, s, statistical, sp, spike_magnitude, energy), dim=-1)
        out = self.fc1(out)
        out = torch.cat((out, energy), dim=-1)
        return self.fc2(out)

In [5]:
best_gen_config_path = 'Config/wgan_g_config.json'
with open(best_gen_config_path) as f:
    best_gen_config = json.load(f)

best_dis_config_path = 'Config/wgan_d_config.json'
with open(best_dis_config_path) as f:
    best_dis_config = json.load(f)

gen = Generator(**best_gen_config).to(device)
dis = Discriminator(**best_dis_config).to(device)

gen.load_state_dict(torch.load('../../Models/WGAN_generator.pt'))
dis.load_state_dict(torch.load('../../Models/WGAN_discriminator.pt'))

gen.eval()
dis.eval()

Discriminator(
  (Embedding_spike): Embedding(5, 3)
  (Embedding_cluster): Embedding(20, 2)
  (time_net): Sequential(
    (0): Linear(in_features=4, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=8, bias=True)
    (3): ReLU()
  )
  (statistical_net): Sequential(
    (0): Linear(in_features=6, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
  )
  (fc1): Sequential(
    (0): Linear(in_features=40, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=4, bias=True)
    (5): ReLU()
  )
  (fc2): Sequential(
    (0): Linear(in_features=5, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=1, bias=True)
  )
)

In [6]:
best_config_path = 'Config/s_vae_config.json'

with open(best_config_path, 'r') as file:
    best_config = json.load(file)

model_m2s = network.m2s_VAE(**best_config).to(device)

model_m2s.load_state_dict(torch.load('../../Models/s_vae.pt'))

model_m2s.eval()

m2s_VAE(
  (lstm_encoder): m2s_Encoder(
    (lstm): LSTM(1, 8, num_layers=2, batch_first=True, bidirectional=True)
    (fc1): Sequential(
      (0): Linear(in_features=16, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=256, bias=True)
      (3): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (4): ReLU()
    )
    (fc2): Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (2): ReLU()
      (3): Linear(in_features=256, out_features=1024, bias=True)
      (4): ReLU()
    )
  )
  (lstm_decoder): m2s_Decoder(
    (Embedding_spike): Embedding(5, 1)
    (Embedding_cluster): Embedding(20, 2)
    (lstm): LSTM(512, 32, num_layers=3, batch_first=True, dropout=0.2, bidirectional=True)
    (time_net): Sequential(
      (0): Linear(in_features=4, out_features=512, bias=True)
      (1): ReLU()
      (2): Linear(in_features=512, out_features=32, bi

In [7]:
eva_training_loader = utils.get_m2s_loader(1)
# long_fake_energy = torch.Tensor().to(device)
long_real_energy = torch.Tensor().to(device)
vae_long_fake_energy = torch.Tensor().to(device)
gan_long_fake_energy = torch.Tensor().to(device)

for i in range(12):

    weather, cluster, time, statistical, spike, real_energy = next(iter(eva_training_loader)) 

    weather = weather.to(device)
    cluster = cluster.to(device).int()
    time = time.to(device)
    statistical = statistical.to(device)
    spike_type = spike[:, :, 0:1].to(dtype=torch.int32)
    spike_magnitude = spike[:, :, 1:].to(dtype=torch.float32)
    spike_type = spike_type.to(device)
    spike_magnitude = spike_magnitude.to(device)
    real_energy = real_energy.to(device)

    noise = torch.randn(real_energy.shape[0], real_energy.shape[1], 1).to(device)

    vae_profile, mu, logvar = model_m2s(noise, weather, cluster, time, statistical, spike_type, spike_magnitude)
    gan_profile = gen(weather, cluster, time, statistical, spike_type, spike_magnitude)

    vae_long_fake_energy = torch.cat((vae_long_fake_energy, vae_profile), dim=-1)
    gan_long_fake_energy = torch.cat((gan_long_fake_energy, gan_profile), dim=-1)
    long_real_energy = torch.cat((long_real_energy, real_energy), dim=-1)

vae_long_fake_energy = vae_long_fake_energy.flatten().detach().cpu().numpy()
gan_long_fake_energy = gan_long_fake_energy.flatten().detach().cpu().numpy()
long_real_energy = long_real_energy.flatten().detach().cpu().numpy()

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=np.arange(0, len(long_real_energy)),
                          y=long_real_energy,
                          mode='lines',
                          name='Real',
                          line=dict(color='blue')))
fig1.show()

# Second figure
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=np.arange(0, len(gan_long_fake_energy)),
                          y=gan_long_fake_energy,
                          mode='lines',
                          name='WGAN-GP',
                          line=dict(color='orange')))
fig2.show()

# Third figure
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=np.arange(0, len(vae_long_fake_energy)),
                          y=vae_long_fake_energy,
                          mode='lines',
                          name='Single VAE',
                          line=dict(color='green')))
fig3.show()

In [8]:
eva_training_loader = utils.get_m2s_loader(1)
long_fake_energy = torch.Tensor().to(device)
long_real_energy = torch.Tensor().to(device)

# For longer sequences
for i in range(12):
    weather, cluster, time, statistical, spike, real_energy = next(iter(eva_training_loader))
    
    weather = weather.to(device)
    cluster = cluster.to(device).int()
    time = time.to(device)
    statistical = statistical.to(device)
    spike_type = spike[:, :, 0:1].to(dtype=torch.int32)  # or torch.long for indexing
    spike_magnitude = spike[:, :, 1:].to(dtype=torch.float32)
    spike_type = spike_type.to(device)
    spike_magnitude = spike_magnitude.to(device)
    real_energy = real_energy.to(device)

    fake_energy = gen(weather, cluster, time, statistical, spike_type, spike_magnitude)
    fake_energy = fake_energy.to(device)

    long_fake_energy = torch.cat((long_fake_energy, fake_energy), dim=-1)
    
    long_real_energy = torch.cat((long_real_energy, real_energy), dim=-1)

long_fake_energy = long_fake_energy.flatten().detach().cpu().numpy()
long_real_energy = long_real_energy.flatten().detach().cpu().numpy()

# plot the results
fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, 200), y=long_real_energy, mode='lines', name='Real', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=np.arange(0, 200), y=long_fake_energy, mode='lines', name='Generated', line=dict(color='orange')))

fig.show()

In [9]:
eva_training_loader = utils.get_m2s_loader(1)
long_fake_energy = torch.Tensor().to(device)
long_real_energy = torch.Tensor().to(device)

for i in range(12):

    weather, cluster, time, statistical, spike, real_energy = next(iter(eva_training_loader)) 

    weather = weather.to(device)
    cluster = cluster.to(device).int()
    time = time.to(device)
    statistical = statistical.to(device)
    spike_type = spike[:, :, 0:1].to(dtype=torch.int32)
    spike_magnitude = spike[:, :, 1:].to(dtype=torch.float32)
    spike_type = spike_type.to(device)
    spike_magnitude = spike_magnitude.to(device)
    real_energy = real_energy.to(device)

    noise = torch.randn(real_energy.shape[0], real_energy.shape[1], 1).to(device)

    synthetic_profile, mu, logvar = model_m2s(noise, weather, cluster, time, statistical, spike_type, spike_magnitude)

    long_fake_energy = torch.cat((long_fake_energy, synthetic_profile), dim=-1)
    long_real_energy = torch.cat((long_real_energy, real_energy), dim=-1)

long_fake_energy = long_fake_energy.flatten().detach().cpu().numpy()
long_real_energy = long_real_energy.flatten().detach().cpu().numpy()

fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, len(long_real_energy)), y=long_real_energy, mode='lines', name='Real', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=np.arange(0, len(long_fake_energy)), y=long_fake_energy, mode='lines', name='Synthetic', line=dict(color='orange')))

fig.show()

In [10]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from ipywidgets import VBox, HBox, Button, IntSlider, SelectionSlider
from IPython.display import display,clear_output
from datetime import date, timedelta, datetime
import plotly.graph_objects as go
import utils
import network
import torch
import json
import joblib

energy_df = pd.read_csv('../Data_preprocess/energy_df.csv')

weather_df = pd.read_csv("../../Data_raw/weather_hourly_darksky.csv")
weather_df['time'] = pd.to_datetime(weather_df['time'])
statistics_df = pd.read_csv('../../Data_processed/Statistics.csv')

fake_energy = []

energy_df['tstp'] = pd.to_datetime(energy_df['tstp'])

ids = statistics_df['LCLid'].unique()

start_date_range = datetime(2013, 1, 1)
end_date_range = datetime(2013, 12, 31)

vae_mean_diff = []
vae_median_diff = []
vae_std_diff = []
vae_min_diff = []
vae_max_diff = []
vae_gradient_diff = []

gan_mean_diff = []
gan_median_diff = []
gan_std_diff = []
gan_min_diff = []   
gan_max_diff = []
gan_gradient_diff = []

vae_mean_diff_percent = []
vae_std_diff_percent = []
vae_median_diff_percent = []
vae_min_diff_percent = []
vae_max_diff_percent = []
vae_gradient_diff_percent = []

gan_mean_diff_percent = []
gan_std_diff_percent = []
gan_median_diff_percent = []
gan_min_diff_percent = []
gan_max_diff_percent = []
gan_gradient_diff_percent = []

vae_kld = []
gan_kld = []

In [13]:
def generate_synthetic_energy_vae_gan(start_date_time = None, end_date_time = None, statistics = None, spike_hours = None, pre_spike_length = None, post_spike_length = None, model_m2s = None, device = None):
    '''
    Generate synthetic energy data

    start_date_time: string with the start date and time in the format 'YYYY-MM-DD HH:MM:SS'
    end_date_time: string with the end date and time in the format 'YYYY-MM-DD HH:MM:SS'

    statistics: list with the statistics in the following order: mean, median, std, min, max, gradient

    spike_hours: list of strings in the format 'HH:MM:SS'
    spike_durations: list of integers
    Should be the same length

    Returns a DataFrame with the time, and the synthetic energy data
    '''
    weather_df = utils.weather_info(start_date_time, end_date_time)
    time_df = weather_df[['tstp']]

    if spike_hours and pre_spike_length and post_spike_length:
        spike_magnitude = statistics[4]
        spike_df = utils.get_spike_df_input(spike_hours, pre_spike_length, post_spike_length, spike_magnitude)
        w_spike_df = utils.merge_weather_spike(weather_df, spike_df)
    else:
        mean = statistics[0]
        std  = statistics[2]
        max = statistics[4]
        statistics_df = pd.DataFrame({'mean': statistics[0], 'median': statistics[1], 'std': statistics[2], 'min': statistics[3], 'max': statistics[4], 'gradient': statistics[5]}, index=[0])
        X = statistics_df.values
        kmoid_cluster = joblib.load('../Data_preprocess/kmedoids_model.joblib')
        cluster = kmoid_cluster.predict(X)[0]
        month = int(start_date_time.split('-')[1])

        start_date_str = start_date_time.split()[0]
        end_date_str = end_date_time.split()[0]

        # Convert the date strings to datetime objects
        start_date = datetime.strptime(start_date_str, '%Y-%m-%d')
        end_date = datetime.strptime(end_date_str, '%Y-%m-%d')

        # Calculate the difference in days
        num_days = (end_date - start_date).days + 1  # Including both start and end dates

        spike_df_days = []
        
        for i in range(num_days):
            spike_df = utils.get_spike_df(cluster, month, mean, std, max)
            spike_df_days.append(spike_df)

        spike_df_days = pd.concat(spike_df_days, ignore_index=True)
        # spike_df_days.to_csv('spike_df_days.csv', index=False)
        
        spike_df_days = utils.add_missing_pre_post_spikes(spike_df_days)
        # spike_df_days.to_csv('spike_df_days_added.csv', index=False)

        w_spike_df = utils.trim_and_merge_spike_weather(weather_df, spike_df_days)
        # w_spike_df.to_csv('w_spike_df.csv', index=False)
        
    w_spike_df = utils.cyclic_time(w_spike_df)

    w_spike_s_df = utils.merge_statistics(w_spike_df, statistics)

    noise = torch.randn(1, len(w_spike_s_df), 1).to(device)

    weather_columns = ['temperature', 'windSpeed', 'humidity']
    cluster_columns = ['kmedoid_clusters']
    time_columns = ['date_sin', 'date_cos', 'time_sin', 'time_cos']
    statistical_columns = ['mean', 'median', 'std',  'min', 'max', 'gradient']
    spike_columns = ['spike_type', 'spike_magnitude']

    weather = torch.tensor(w_spike_s_df[weather_columns].values, dtype=torch.float32).unsqueeze(0).to(device)
    cluster = torch.tensor(w_spike_s_df[cluster_columns].values, dtype=torch.float32).unsqueeze(0).to(device).int()
    time = torch.tensor(w_spike_s_df[time_columns].values, dtype=torch.float32).unsqueeze(0).to(device)
    statistical = torch.tensor(w_spike_s_df[statistical_columns].values, dtype=torch.float32).unsqueeze(0).to(device)
    spike = torch.tensor(w_spike_s_df[spike_columns].values, dtype=torch.float32).unsqueeze(0).to(device).int()

    vae_synthetic_energy, mu, logvar = model_m2s(noise, weather, cluster, time, statistical, spike[:, :, 0:1].int(), spike[:, :, 1:])
    vae_synthetic_energy = vae_synthetic_energy.squeeze().detach().cpu().numpy()

    gan_synthetic_energy = gen(weather, cluster, time, statistical, spike[:, :, 0:1].int(), spike[:, :, 1:]).squeeze().detach().cpu().numpy()

    return vae_synthetic_energy, gan_synthetic_energy

In [15]:
for i in range(10):
    selected_id = np.random.choice(ids, 1)
    real_profile = energy_df[energy_df['LCLid'] == selected_id.item()]
    temp_df = statistics_df[statistics_df['LCLid'].isin(selected_id)].iloc[0]
    
    mean_v= max(0.001, temp_df['mean'] + np.random.uniform(-0.05, 0.05))
    mean_v = round(mean_v, 4)
    median_v = max(0.001,temp_df['median'] + np.random.uniform(-0.05, 0.05))
    median_v = round(median_v, 4)
    std_v= max(0.001,temp_df['std'] + np.random.uniform(-0.05, 0.05))
    std_v = round(std_v, 4)
    min_v = max(0.001,temp_df['min'] + np.random.uniform(-0.01, 0.01))
    min_v = round(min_v, 4)
    max_v = max(0.001,temp_df['max'] + np.random.uniform(-0.05, 0.05))
    max_v = round(max_v, 4)
    gradient_v = max(0.001,temp_df['gradient'] + np.random.uniform(-0.01, 0.01))
    gradient_v = round(gradient_v, 4)

    statistics = [mean_v, median_v, std_v, min_v, max_v, gradient_v]

    for j in range(10):
        np.random.seed(j)
        random_start_date = start_date_range + timedelta(days=np.random.randint(0, (end_date_range - start_date_range).days - 32))
        # Ensure the end date does not exceed a 4-day length from the start date
        random_end_date = random_start_date + timedelta(days=np.random.randint(1, 30))

        real_profile = energy_df[(energy_df['tstp'] >= random_start_date) & 
                                 (energy_df['tstp'] <= random_end_date) & 
                                 (energy_df['LCLid'] == selected_id.item())]

        real_energy = real_profile['energy(kWh/hh)'].values

        # Some profiles may be empty due to the random date selection
        if len(real_energy) == 0:
            continue

        test_start_date_time = random_start_date.strftime('%Y-%m-%d %H:%M:%S')
        test_end_date_time = random_end_date.strftime('%Y-%m-%d %H:%M:%S')

        
        for k in range(50):
            np.random.seed(k)
            
            print(i, j, k)

            vae_synthetic, gan_synthetic = generate_synthetic_energy_vae_gan(start_date_time=test_start_date_time, 
                                                        end_date_time=test_end_date_time, 
                                                        statistics=statistics, 
                                                        model_m2s=model_m2s, 
                                                        device=device)
            print(i, j, k)

            synthetic_len = len(vae_synthetic)
            real_len = len(real_energy)

            if synthetic_len > real_len:
                vae_synthetic = vae_synthetic[:real_len]
                gan_synthetic = gan_synthetic[:real_len]
            elif synthetic_len < real_len:
                real_energy = real_energy[:synthetic_len]

            vae_mean = np.mean(vae_synthetic)
            vae_median = np.median(vae_synthetic)
            vae_std = np.std(vae_synthetic)
            vae_min = np.min(vae_synthetic)
            vae_max = np.max(vae_synthetic)
            vae_gradient = np.mean(abs(vae_synthetic[:-1] - vae_synthetic[1:]))
            
            gan_mean = np.mean(gan_synthetic)
            gan_median = np.median(gan_synthetic)
            gan_std = np.std(gan_synthetic)
            gan_min = np.min(gan_synthetic)
            gan_max = np.max(gan_synthetic)
            gan_gradient = np.mean(abs(gan_synthetic[:-1] - gan_synthetic[1:]))
            
            vae_mean_diff.append(abs(mean_v - vae_mean))
            vae_median_diff.append(abs(median_v - vae_median))
            vae_std_diff.append(abs(std_v - vae_std))
            vae_min_diff.append(abs(min_v - vae_min))
            vae_max_diff.append(abs(max_v - vae_max))
            vae_gradient_diff.append(abs(gradient_v - vae_gradient))

            gan_mean_diff.append(abs(mean_v - gan_mean))
            gan_median_diff.append(abs(median_v - gan_median))
            gan_std_diff.append(abs(std_v - gan_std))
            gan_min_diff.append(abs(min_v - gan_min))
            gan_max_diff.append(abs(max_v - gan_max))
            gan_gradient_diff.append(abs(gradient_v - gan_gradient))

            vae_mean_diff_percent.append(abs(mean_v - vae_mean) / mean_v)
            vae_median_diff_percent.append(abs(median_v - vae_median) / median_v)
            vae_std_diff_percent.append(abs(std_v - vae_std) / std_v)
            vae_min_diff_percent.append(abs(min_v - vae_min) / min_v)
            vae_max_diff_percent.append(abs(max_v - vae_max) / max_v)
            vae_gradient_diff_percent.append(abs(gradient_v - vae_gradient) / gradient_v)

            gan_mean_diff_percent.append(abs(mean_v - gan_mean) / mean_v)
            gan_median_diff_percent.append(abs(median_v - gan_median) / median_v)
            gan_std_diff_percent.append(abs(std_v - gan_std) / std_v)
            gan_min_diff_percent.append(abs(min_v - gan_min) / min_v)
            gan_max_diff_percent.append(abs(max_v - gan_max) / max_v)
            gan_gradient_diff_percent.append(abs(gradient_v - gan_gradient) / gradient_v)

            vae_kld.append(torch.nn.functional.kl_div(torch.tensor(vae_synthetic), torch.tensor(real_energy), reduction='batchmean'))
            gan_kld.append(torch.nn.functional.kl_div(torch.tensor(gan_synthetic), torch.tensor(real_energy), reduction='batchmean'))

print('Evaluation Results for VAE and GAN Synthetic Profiles:')
print('-----------------------------------------------')
print('Mean error for VAE synthetic profile:', np.mean(vae_mean_diff))
print('Median error for VAE synthetic profile:', np.mean(vae_median_diff))
print('Std error for VAE synthetic profile:', np.mean(vae_std_diff))
print('Min error for VAE synthetic profile:', np.mean(vae_min_diff))
print('Max error for VAE synthetic profile:', np.mean(vae_max_diff))
print('Gradient error for VAE synthetic profile:', np.mean(vae_gradient_diff))

print('Mean error for GAN synthetic profile:', np.mean(gan_mean_diff))
print('Median error for GAN synthetic profile:', np.mean(gan_median_diff))
print('Std error for GAN synthetic profile:', np.mean(gan_std_diff))
print('Min error for GAN synthetic profile:', np.mean(gan_min_diff))
print('Max error for GAN synthetic profile:', np.mean(gan_max_diff))
print('Gradient error for GAN synthetic profile:', np.mean(gan_gradient_diff))

print('Mean Percentage error for VAE synthetic profile:', np.mean(vae_mean_diff_percent))
print('Median Percentage error for VAE synthetic profile:', np.mean(vae_median_diff_percent))
print('Std Percentage error for VAE synthetic profile:', np.mean(vae_std_diff_percent))
print('Min Percentage error for VAE synthetic profile:', np.mean(vae_min_diff_percent))
print('Max Percentage error for VAE synthetic profile:', np.mean(vae_max_diff_percent))
print('Gradient Percentage error for VAE synthetic profile:', np.mean(vae_gradient_diff_percent))

print('Mean Percentage error for GAN synthetic profile:', np.mean(gan_mean_diff_percent))
print('Median Percentage error for GAN synthetic profile:', np.mean(gan_median_diff_percent))
print('Std Percentage error for GAN synthetic profile:', np.mean(gan_std_diff_percent))
print('Min Percentage error for GAN synthetic profile:', np.mean(gan_min_diff_percent))
print('Max Percentage error for GAN synthetic profile:', np.mean(gan_max_diff_percent))
print('Gradient Percentage error for GAN synthetic profile:', np.mean(gan_gradient_diff_percent))

0 0 0
0 0 0
0 0 1
0 0 1
0 0 2
0 0 2
0 0 3
0 0 3
0 0 4
0 0 4
0 0 5
Patience for arranging spike times exceeded, use the last valid time.
0 0 5
0 0 6
0 0 6
0 0 7
0 0 7
0 0 8
0 0 8
0 0 9
0 0 9
0 0 10
0 0 10
0 0 11
0 0 11
0 0 12
0 0 12
0 0 13
0 0 13
0 0 14
0 0 14
0 0 15
0 0 15
0 0 16
0 0 16
0 0 17
0 0 17
0 0 18
0 0 18
0 0 19
0 0 19
0 0 20
0 0 20
0 0 21
0 0 21
0 0 22
0 0 22
0 0 23
0 0 23
0 0 24
0 0 24
0 0 25
0 0 25
0 0 26
0 0 26
0 0 27
0 0 27
0 0 28
0 0 28
0 0 29
0 0 29
0 0 30
0 0 30
0 0 31
0 0 31
0 0 32
Patience for arranging spike times exceeded, use the last valid time.
0 0 32
0 0 33
0 0 33
0 0 34
0 0 34
0 0 35
0 0 35
0 0 36
0 0 36
0 0 37
0 0 37
0 0 38
0 0 38
0 0 39
0 0 39
0 0 40
0 0 40
0 0 41
Patience for arranging spike times exceeded, use the last valid time.
0 0 41
0 0 42
0 0 42
0 0 43
0 0 43
0 0 44
0 0 44
0 0 45
0 0 45
0 0 46
0 0 46
0 0 47
0 0 47
0 0 48
0 0 48
0 0 49
0 0 49
0 1 0
0 1 0
0 1 1
0 1 1
0 1 2
0 1 2
0 1 3
0 1 3
0 1 4
0 1 4
0 1 5
0 1 5
0 1 6
0 1 6
0 1 7
0 1 7
0 1 8
0 1 8
0 

In [16]:
print('KLD for VAE synthetic profile:', np.mean(vae_kld))
print('KLD for GAN synthetic profile:', np.mean(gan_kld))

KLD for VAE synthetic profile: -0.2992774126303226
KLD for GAN synthetic profile: -0.37372770252198706
